#### Импорт библиотек

In [37]:
from bisect import bisect_left, bisect_right
import random
import math
from typing import Optional

#### Функция для вычисления НОД

In [38]:
def gcd(a, b):
    a, b = abs(a), abs(b)
    while b != 0:
        a, b = b, a % b
    return a

#### Функция для построения таблицы простых чисел до C

In [39]:
# создание решета Эратосфена
def create_sieve(n: int) -> list[int]:
    if n > 10**6:
        print("n должно быть <= 10**6")
        return None
        
    if n < 2:
        return []
        
    is_prime = bytearray(b'\x01') * (n + 1)
    is_prime[0:2] = b'\x00\x00'

    for p in range(2, int(n**0.5) + 1):
        if is_prime[p]:
            is_prime[p*p : n+1 : p] = b'\x00' * len(range(p*p, n+1, p))
            
    return [i for i, flag in enumerate(is_prime) if flag]

#### Тест Миллера-Рабина

In [40]:
def test_miller_rabin(n: int, rounds: int = 20) -> bool:
    # находим d, s
    d = n - 1
    s = 0

    while d % 2 == 0:
        d //= 2
        s += 1

    # попытки теста
    for _ in range(rounds):
        # случайное основание a
        a = random.randint(2, n - 2)

        g = gcd(a, n)
        if g > 1:
            return False

        # вычисляем b = a^d mod n
        b = pow(a, d, n)

        if b == 1 or b == n - 1:
            continue

        passed_round = False

        # проверка цепочки b^2 ...
        for _ in range(s - 1):
            b = pow(b, 2, n)

            if b == n - 1:
                passed_round = True
                break

            if b == 1:
                return False

        if not passed_round:
            return False

    return True

#### Тест Лемера

In [41]:
def test_lehmer(p: int, q: int, rounds: int = 20) -> bool:

    k = (p - 1) // q

    for _ in range(rounds):
        a = random.randint(2, p - 2)

        if gcd(a, p) != 1:
            return False

        # первое условие
        if pow(a, p - 1, p) != 1:
            return False

        # второе условие
        if gcd(pow(a, k, p) - 1, p) == 1:
            return True

    return False

#### Организация ввода границ A и B для алгоритма

In [42]:
def input_limits() -> list[int]:
    while True:
        try:
            print("Введите границу A:")
            A = int(input())
            print("Введите границу B:")
            B = int(input())

            if A <= 0 or B <= 0:
                print("Границы A и B должны быть натуральными")
                continue
            if B <= A + 1:
                print("Границы должны удовлетворять условию: B > A + 1")
                continue

            return [A, B] 

        except ValueError:
            print("Введите целые числа")

#### Алгоритм генерации простого числа рекурсивным методом

In [43]:
def generate_prime(A: int, B: int, C: int, sieve: list[int], max_q_attempts: int) -> Optional[int]:

    # если B<=C, то простое число p выбирается случайно (A < p < B)
    if B <= C:
        left = bisect_right(sieve, A)
        right = bisect_left(sieve, B)
        if left >= right:
            print(f"В диапазоне {A} < p < {B} нет подходящих чисел")
            return None
        return random.choice(sieve[left:right])

    # параметры алгоритма
    # q_A = ceil(sqrt(B))
    q_A = math.isqrt(B)
    if q_A * q_A < B:
        q_A += 1
    
    # q_B = floor(alpha * q_A), alpha = (B + A - 1) / (2A)
    q_B = ((B + A - 1) * q_A) // (2 * A)
    
    # k_start = ceil(A / q_A)
    k_start = (A + q_A - 1) // q_A
    
    # k_end = floor((B - 1) / q_B)
    k_end = (B - 1) // q_B
    
    if k_start % 2 != 0:
        k_start += 1
    if k_end % 2 != 0:
        k_end -= 1
        
    if (k_start > k_end) or (q_B <= q_A + 1):
        return None

    small_primes = [3, 5, 7, 11, 13, 17, 19, 23, 29, 31]
    
    for _ in range(max_q_attempts):
        # генерация простого q 
        q = generate_prime(q_A, q_B, C, sieve, max_q_attempts)
        if q is None:
            continue

        # первый кандидат p
        k = k_start
        p = q * k + 1

        # остатки p mod d_i
        deltas = [p % d for d in small_primes]
        steps = [(2 * q) % d for d in small_primes]
        
        # поиск простого p
        while k <= k_end:
            # проверка делимости на малые простые
            if all(delta != 0 for delta in deltas):

                # тест Миллера-Рабина
                if test_miller_rabin(p):

                    # тест Лемера
                    if test_lehmer(p, q):
                        return p

            k = k + 2
            p = p + 2 * q

            # обновление остатков
            for i, d in enumerate(small_primes):
                deltas[i] += steps[i]
                if deltas[i] >= d:
                    deltas[i] -= d

    return None

#### Пример использования

In [54]:
C = 2**16
A, B = input_limits()
sieve = create_sieve(C)
max_q_attempts = 20
prime = generate_prime(A, B, C, sieve, max_q_attempts)
print("Простое число: ", prime)

Введите границу A:


 33


Введите границу B:


 34


Границы должны удовлетворять условию: B > A + 1
Введите границу A:


 4


Введите границу B:


 10


Простое число:  7
